In [26]:
from google.colab import userdata
import os

os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')



In [27]:
# Téléchargement et extraction dans un dossier 'aircraft_data'
!kaggle datasets download -d seryouxblaster764/fgvc-aircraft --unzip -p ./aircraft_data

Dataset URL: https://www.kaggle.com/datasets/seryouxblaster764/fgvc-aircraft
License(s): other
 99% 2.55G/2.57G [00:37<00:00, 247MB/s]
100% 2.57G/2.57G [00:37<00:00, 73.3MB/s]


In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col, concat, lit

spark = SparkSession.builder.appName("AircraftLabels").getOrCreate()

# 1. Charger le fichier des constructeurs
# Assure-toi du nom exact du fichier dans aircraft_data/data/
path_manufacturers = "./aircraft_data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data/images_manufacturer_test.txt"
raw_df = spark.read.text(path_manufacturers)

# 2. Séparer l'ID du constructeur
# On split sur le premier espace trouvé
df_labels = raw_df.select(
    split(col("value"), " ", 2).getItem(0).alias("image_id"),
    split(col("value"), " ", 2).getItem(1).alias("manufacturer")
)

# 3. Créer le chemin complet vers l'image .jpg
df_final = df_labels.withColumn(
    "path",
    concat(lit("./aircraft_data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data/images/"), col("image_id"), lit(".jpg"))
)

df_final.show(5)

+--------+------------+--------------------+
|image_id|manufacturer|                path|
+--------+------------+--------------------+
| 1514522|      Boeing|./aircraft_data/f...|
| 0747566|      Boeing|./aircraft_data/f...|
| 1008575|      Boeing|./aircraft_data/f...|
| 0717480|      Boeing|./aircraft_data/f...|
| 0991569|      Boeing|./aircraft_data/f...|
+--------+------------+--------------------+
only showing top 5 rows


In [49]:
from pyspark.ml.feature import StringIndexer

# Initialiser l'indexer
indexer = StringIndexer(inputCol="manufacturer", outputCol="label")

# L'ajuster aux données et transformer le DataFrame
df_indexed = indexer.fit(df_final).transform(df_final)

df_indexed.select("image_id", "manufacturer", "label").show(10)

+--------+------------+-----+
|image_id|manufacturer|label|
+--------+------------+-----+
| 1514522|      Boeing|  0.0|
| 0747566|      Boeing|  0.0|
| 1008575|      Boeing|  0.0|
| 0717480|      Boeing|  0.0|
| 0991569|      Boeing|  0.0|
| 1446335|      Boeing|  0.0|
| 1345732|      Boeing|  0.0|
| 0064932|      Boeing|  0.0|
| 1453508|      Boeing|  0.0|
| 0536515|      Boeing|  0.0|
+--------+------------+-----+
only showing top 10 rows


In [50]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

# Charger le modèle pré-entraîné
model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

# On diffuse les poids du modèle sur tous les nœuds du cluster Spark
bc_model_weights = spark.sparkContext.broadcast(model.get_weights())

def get_model():
    """Fonction pour reconstruire le modèle sur chaque worker Spark"""
    model = ResNet50(weights=None, include_top=False, pooling='avg')
    model.set_weights(bc_model_weights.value)
    return model

@pandas_udf("array<float>")
def featurize_series(content_series: pd.Series) -> pd.Series:
    model = get_model()
    features_list = []

    for path in content_series:
        # 1. Charger et redimensionner l'image pour ResNet (224x224)
        img = load_img(path, target_size=(224, 224))
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        # 2. Extraire les caractéristiques
        feat = model.predict(x)
        features_list.append(feat.flatten())

    return pd.Series(features_list)
# Attention : cette étape peut prendre quelques minutes selon le nombre d'images
df_features = df_indexed.withColumn("features", featurize_series("path"))

# On vérifie le résultat
df_features.select("manufacturer", "label", "features").show(5)

+------------+-----+--------------------+
|manufacturer|label|            features|
+------------+-----+--------------------+
|      Boeing|  0.0|[0.010274294, 0.4...|
|      Boeing|  0.0|[0.023007687, 0.0...|
|      Boeing|  0.0|[0.074908905, 0.1...|
|      Boeing|  0.0|[0.16759637, 0.05...|
|      Boeing|  0.0|[0.38661242, 0.18...|
+------------+-----+--------------------+
only showing top 5 rows
